# This example script uses the data to identify possible district that have high rice/wheat rotation GHG emissions for reduction programs.


In [2]:
%reload_ext autoreload
%autoreload 2

get_ipython().run_line_magic('matplotlib', 'inline')
import sys
sys.path.extend(["..\\"])
from Shared import NIBSData2
from Shared.NIBSData2 import *
import duckdb
import geopandas as gpd
import warnings
# Suppress all warnings
warnings.filterwarnings("ignore")


nibs = NIBSData2.NIBSData()

## create in-memory duckdb database of needed parquet table files. The analysis needs the following tables:
- nitrogen_universe
- data_boundaries
- vwA_apy_adj_crop_area_yield
- n2o_total_direct_n_balance_eagle_2020_summary
-state_codes
The below code demostrates how to use the helper class `NIBSData2` to load data and get the

In [ ]:
# Create an in-memory DuckDB connection
con = duckdb.connect(database=":memory:")
con.execute("install spatial")
con.execute("load spatial")
con.execute("SET default_collation = 'nocase';")

# Tables/views listed in the notebook markdown
required_parquets = [
    "nitrogen_universe",
    # "data_boundaries",
    "vwA_apy_adj_crop_area_yield",
    "n2o_total_direct_n_balance_eagle_2020_summary",
    "state_codes",
]

loaded = []
missing = []

# load non-spatial tables
for name in required_parquets:
    url = nib_urls.get(name)
    if url:
        con.execute(f'''
            CREATE OR REPLACE TABLE "{name}" AS
            SELECT * FROM read_parquet('{url}')
        ''')
        loaded.append(name)
    else:
        missing.append(name)

# load spatial table
url = nib_urls.get("data_boundaries")
if url:
    con.execute(f'''
        CREATE OR REPLACE TABLE "data_boundaries" AS
        SELECT * EXCLUDE geog, (CAST(geog AS VARCHAR)) AS geog FROM read_parquet('{url}')
    ''')
    loaded.append("data_boundaries")
else:
    missing.append("data_boundaries")
 
print("Loaded:", loaded)   
if missing:
    print("Missing in nib_urls:", missing)

# Optional: verify what is available in the in-memory database
con.sql("SHOW TABLES").df()

Loaded: ['nitrogen_universe', 'vwA_apy_adj_crop_area_yield', 'n2o_total_direct_n_balance_eagle_2020_summary', 'state_codes']


,name
0,data_boundaries
1,n2o_total_direct_n_balance_eagle_2020_summary
2,nitrogen_universe
3,state_codes
4,vwA_apy_adj_crop_area_yield


Build dataset of district that grow both rice adn wheat. 

In [43]:
sql = """
with a as
	(
	select 
 		r.link_id as rice_id, w.link_id as wheat_id, r.crop, r.apy_crop as rice_crop, w.apy_crop as wheat_crop
		, ROW_NUMBER() over(order by w.link_id) as rotation_id
		,ROW_NUMBER() over(partition by r.geog_checksum, r.IRRIGATION_STATUS, r.SIZE_GROUPHA order by len(r.crop)) as dups
		,concat(district_name,'(',state_code,')') as label
		,state_code 
	from nitrogen_universe as r 
		join nitrogen_universe as w on r.geog_checksum = w.geog_checksum 
			and r.SIZE_GROUPHA = w.SIZE_GROUPHA
			and r.IRRIGATION_STATUS = w.IRRIGATION_STATUS
			and lower(r.apy_crop) = 'rice'
			and lower(w.apy_crop) = 'wheat'
		join data_boundaries as db on db.geog_checksum = r.geog_checksum
			and db.geog_checksum = w.geog_checksum
		join state_codes as sc on lower(sc.state_name) = lower(db.state_name) 
	)
,b as
	(
	select u.link_id
		,a.rotation_id
		, u.geog_checksum
		, label
		,state_code 
		, u.apy_crop
		, farm_size 
		, irrigated 
		, u.adj_crop_area
		, u.adj_fert_area
		, u.mean_yield as mean_yield_t_ha
		, u.sd_yield as sd_yield_t_ha
		,mean_total_n_balance_n_kg_ha
		,sd_total_n_balance_n_kg_ha 
	from vwA_apy_adj_crop_area_yield as u 
		join n2o_total_direct_n_balance_eagle_2020_summary as n on u.link_id = n.link_id
		join a on a.rice_id = u.link_id
	where dups = 1
	union
	select u.link_id
		,a.rotation_id
		, u.geog_checksum
		, label
		,state_code 
		, u.apy_crop
		, farm_size 
		, irrigated 
		, u.adj_crop_area
		, u.adj_fert_area
		, u.mean_yield as mean_yield_t_ha
		, u.sd_yield as sd_yield_t_ha
		,mean_total_n_balance_n_kg_ha
		,sd_total_n_balance_n_kg_ha 
	from vwA_apy_adj_crop_area_yield as u 
		join n2o_total_direct_n_balance_eagle_2020_summary as n on u.link_id = n.link_id
		join a on a.wheat_id = u.link_id
	where dups = 1
	)
,c as
	(
	select *
		,count(*) over(partition by rotation_id) as rcnt
	from b 
	)
select c.*
	,geog
	,label_x
	,label_y 
from c
	join data_boundaries as d on d.geog_checksum = c.geog_checksum
where rcnt = 2;
""" 

# table = ['nitrogen_universe', 'data_boundaries', 'vwA_apy_adj_crop_area_yield', 'n2o_total_direct_n_balance_eagle_2020_summary', 'state_codes']

# for t in table:
# 	print(f"Sample from {t}:")
# 	print(con.execute(f"SELECT * FROM {t} LIMIT 5").fetch_df())

sql = "select geog from data_boundaries limit 5"

df = con.execute(sql).fetch_df()
# gdf = gpd.GeoDataFrame(df,geometry= gpd.GeoSeries.from_wkt(df['geog']),crs=4326)
df.head(5)

,geog
0,\x02\x04\x00\x00\x00\x00\x00\x00^\xAF\x99B\x00...
1,\x02\x04\x00\x00\x00\x00\x00\x00\x82t\xB6BV\xB...
2,\x02\x04\x00\x00\x00\x00\x00\x00\xF6l\xB8B(\xF...
3,\x05\x04\x00\x00\x00\x00\x00\x00\x17\x1A\x94B\...
4,\x02\x04\x00\x00\x00\x00\x00\x00\x84h\x98B\x81...


In [19]:

# gdf = nibs.load_dataset("data_boundaries")
# gdf.head(5)

with duckdb.connect(database=":memory:") as con:
    con.execute("install spatial")
    con.execute("load spatial")
    df = con.execute("""
                     SELECT *
                     ,  cast(geog as string) as geog_str  
                     FROM read_parquet("C:\\Users\\HP\\npongo Dropbox\\benjamin clark\\CIL\\Products\\ShareableData\\duckdb_to_parquet\\vwM_district_n2o_co2e_upland_crop_results_shcherbak_2014.parquet")
                     OFFSET 100 LIMIT 1
                     """).fetch_df()
df

# gdf = gpd.GeoDataFrame(df,geometry= gpd.GeoSeries.from_wkt(df['geog_str']),crs=4326)
# gdf.head(5)

,gwp_time_period,geog_checksum,crop,model,emission,label,adj_crop_area,direct_kg_n2o_co2e_ha,direct_sd_kg_n2o_co2e_ha,kg_n2o_co2e_ha,...,total_t_co2e,sd_total_t_co2e,direct_t_co2e,sd_direct_t_co2e,total_Gg_co2e_map,sd_total_Gg_co2e_map,direct_Gg_co2e_map,sd_direct_Gg_co2e_map,geog,geog_str
0,20,-1489678967,Upland crops,"Shcherbak et al., 2014",N2O,$N_2O_{CO_2e_{20}}$,23513.0,48.548583,17.485936,560.651345,...,13182.595084,1459.699466,1141.522829,411.14681,13.182595,1.459699,1.141523,0.411147,"[2, 4, 0, 0, 0, 0, 0, 0, 54, 242, 167, 66, 115...","POLYGON ((84.7429713 24.0480777, 84.7392645 24..."


In [29]:
import duckdb 

with duckdb.connect(r"b:\repos\nibs\data\india_agriculture_census_ghg_results_v2.duckdb") as conn:
    sql = "select * from data_boundaries"
    sql = "SHOW ALL TABLES;"
    df = conn.execute(sql).fetchall()
    rows=conn.execute('select table_schema, table_name from information_schema.tables where table_schema not in (\'information_schema\', \'pg_catalog\') order by table_schema, table_name').fetchall()
    conn.close


rows

[]